# Tugas JS03 PBL - Perbandingan Model Sleep Wise

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv(Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv'))
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')
bp = df['Blood Pressure'].str.split('/', expand=True).astype(float)
df['Systolic BP'], df['Diastolic BP'] = bp[0], bp[1]
df = df.drop(columns=['Person ID', 'Blood Pressure'])
X, y = df.drop(columns='Sleep Disorder'), df['Sleep Disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
num = X.select_dtypes(include=['number']).columns.tolist()
cat = [col for col in X.columns if col not in num]
prep = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
models = {'Logistic Regression': LogisticRegression(max_iter=1000), 'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42), 'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')}
results = {}
trained_pipelines = {}
for name, classifier in models.items():
    pipe = Pipeline([('prep', prep), ('classifier', classifier)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    trained_pipelines[name] = pipe
    results[name] = accuracy_score(y_test, pred)
    print(f'{name}: {results[name]:.2%}')
best_name = max(results, key=results.get)
with open('js03pbl_best_model.pkl', 'wb') as file: pickle.dump(trained_pipelines[best_name], file)
print('\nModel terbaik:', best_name)

Logistic Regression: 93.33%
Decision Tree: 93.33%


Random Forest: 86.67%

Model terbaik: Logistic Regression
